In [1]:
from roboflow import Roboflow
import os

API_KEY = "CfVGS82wt63mHysL62v4"
DATASET_PATH = r"d:\Dicoding\capstone-project\SIDIAS-\data_science\data\processed\gambar"


# Buat folder baru
os.makedirs(DATASET_PATH, exist_ok=True)

# Download dengan overwrite
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("mnt-bgmps").project("deteksi-stunting-jj1oq")
dataset = project.version(1).download("coco", location=DATASET_PATH, overwrite=True)

print(f"\n✓ Download selesai!")
print(f"Dataset ada di: {DATASET_PATH}")

# Verifikasi
print(f"\n=== Verifikasi ===")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root) or 'dataset_gambar'
    print(f"{indent}📁 {folder}/")
    
    subindent = '  ' * (level + 1)
    for f in files[:5]:
        print(f"{subindent}📄 {f}")
    if len(files) > 5:
        print(f"{subindent}... dan {len(files)-5} file lainnya")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to d:\Dicoding\capstone-project\SIDIAS-\data_science\data\processed\gambar in coco:: 100%|██████████| 1861/1861 [00:00<00:00, 2727.95it/s]


✓ Download selesai!
Dataset ada di: d:\Dicoding\capstone-project\SIDIAS-\data_science\data\processed\gambar

=== Verifikasi ===
📁 gambar/
  📄 README.dataset.txt
  📄 README.roboflow.txt
  📁 test/
    📄 Malnurished-135-_jpg.rf.a25639f5b696a1dd5cfff1d4be39e4bb.jpg
    📄 Malnurished-14-_jpg.rf.986b73314b295d7094eb1f89a00668b2.jpg
    📄 Malnurished-152-_jpg.rf.d29c4897af9102370263a955c698e490.jpg
    📄 Malnurished-153-_jpg.rf.59523cabdb710a7a222a713e32df0f3e.jpg
    📄 Malnurished-154-_jpg.rf.546e431b5fef76d3cd2844ddb45f1d99.jpg
    ... dan 182 file lainnya
  📁 train/
    📄 Malnurished-1-_jpeg.rf.cb8004bb25bfc6205b785ef4036b98dc.jpg
    📄 Malnurished-1-_jpg.rf.739a30fba704bc319a95d1835316b149.jpg
    📄 Malnurished-1-_png.rf.01b94a181d72cd8e0521cf0d9b807e2b.jpg
    📄 Malnurished-1-_webp.rf.f16baef35ae422da18658e55775f69ae.jpg
    📄 Malnurished-10-_jpeg.rf.b605fcd0385d58786e4973f6116b6592.jpg
    ... dan 1294 file lainnya
  📁 valid/
    📄 Malnurished-10-_jpg.rf.9521b02ff15a8580ed7ca6397137773

In [3]:
import os
import json
import shutil
from tqdm import tqdm

def organize_dataset(target_folder):
    # 1. Tentukan path folder (train/valid/test)
    folder_path = os.path.join(DATASET_PATH, target_folder)
    
    if not os.path.exists(folder_path):
        print(f"⚠️ Folder {target_folder} tidak ditemukan di path tersebut.")
        return

    # 2. Cari file JSON di dalam folder tersebut
    json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
    
    if not json_files:
        print(f"⚠️ Tidak ada file JSON di dalam folder {target_folder}.")
        return
    
    # Ambil file JSON pertama yang ditemukan
    json_path = os.path.join(folder_path, json_files[0])
    print(f"📄 Menggunakan file anotasi: {json_files[0]} untuk folder {target_folder}")

    # 3. Baca data JSON
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # 4. Ambil mapping kategori
    categories = {cat['id']: cat['name'] for cat in data['categories']}
    print(f"🏷️ Kategori ditemukan: {list(categories.values())}")

    # 5. Buat sub-folder untuk tiap kategori (Healthy, Stunting, dll)
    for cat_name in categories.values():
        os.makedirs(os.path.join(folder_path, cat_name), exist_ok=True)

    # 6. Pindahkan gambar ke folder kategorinya
    images = {img['id']: img['file_name'] for img in data['images']}
    moved_count = 0
    
    for ann in tqdm(data['annotations'], desc=f"Memproses {target_folder}"):
        img_name = images.get(ann['image_id'])
        cat_name = categories.get(ann['category_id'])
        
        src = os.path.join(folder_path, img_name)
        dst = os.path.join(folder_path, cat_name, img_name)
        
        if os.path.exists(src):
            shutil.move(src, dst)
            moved_count += 1

    print(f"✅ Selesai! {moved_count} gambar di folder {target_folder} telah masuk ke sub-folder label.\n")

# Eksekusi untuk ketiga folder utama
for folder in ['train', 'valid', 'test']:
    organize_dataset(folder)

# 7. Cek hasil akhir (Audit Data Science)
print("=== DISTRIBUSI DATA AKHIR ===")
for folder in ['train', 'valid', 'test']:
    p = os.path.join(DATASET_PATH, folder)
    if os.path.exists(p):
        print(f"\n📁 Folder {folder.upper()}:")
        for sub in os.listdir(p):
            sub_path = os.path.join(p, sub)
            if os.path.isdir(sub_path):
                print(f"   - {sub}: {len(os.listdir(sub_path))} gambar")

📄 Menggunakan file anotasi: _annotations.coco.json untuk folder train
🏷️ Kategori ditemukan: ['Deteksi-Stunting', 'Healthy', 'MalNutrisi', 'Stunting']


Memproses train: 100%|██████████| 1175/1175 [00:00<00:00, 25929.18it/s]


✅ Selesai! 0 gambar di folder train telah masuk ke sub-folder label.

📄 Menggunakan file anotasi: _annotations.coco.json untuk folder valid
🏷️ Kategori ditemukan: ['Deteksi-Stunting', 'Healthy', 'MalNutrisi', 'Stunting']


Memproses valid: 100%|██████████| 334/334 [00:00<00:00, 40018.78it/s]


✅ Selesai! 0 gambar di folder valid telah masuk ke sub-folder label.

📄 Menggunakan file anotasi: _annotations.coco.json untuk folder test
🏷️ Kategori ditemukan: ['Deteksi-Stunting', 'Healthy', 'MalNutrisi', 'Stunting']


Memproses test: 100%|██████████| 171/171 [00:00<00:00, 41118.27it/s]

✅ Selesai! 0 gambar di folder test telah masuk ke sub-folder label.

=== DISTRIBUSI DATA AKHIR ===

📁 Folder TRAIN:
   - Deteksi-Stunting: 0 gambar
   - Healthy: 614 gambar
   - MalNutrisi: 24 gambar
   - Stunting: 531 gambar

📁 Folder VALID:
   - Deteksi-Stunting: 0 gambar
   - Healthy: 174 gambar
   - MalNutrisi: 15 gambar
   - Stunting: 145 gambar

📁 Folder TEST:
   - Deteksi-Stunting: 0 gambar
   - Healthy: 86 gambar
   - MalNutrisi: 7 gambar
   - Stunting: 77 gambar


In [8]:
import albumentations as A
import cv2
import os
import glob
from tqdm import tqdm

# 1. Konfigurasi
TARGET_COUNT = 550
folder_to_augment = os.path.join(DATASET_PATH, "train", "MalNutrisi")

# 2. Definisi Teknik Augmentasi (Sederhana tapi Ampuh)
# Kita buat variasi posisi dan cahaya agar model AI tidak bosan
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
])

# 3. Ambil data asli yang cuma 24 biji itu
current_files = glob.glob(os.path.join(folder_to_augment, "*.jpg")) + \
                glob.glob(os.path.join(folder_to_augment, "*.png"))

num_existing = len(current_files)
num_needed = TARGET_COUNT - num_existing

if num_needed > 0:
    print(f"🔥 Menambah {num_needed} gambar augmentasi untuk MalNutrisi...")
    
    generated_count = 0
    pbar = tqdm(total=num_needed)
    
    while generated_count < num_needed:
        for img_path in current_files:
            if generated_count >= num_needed: break
            
            # Baca gambar
            image = cv2.imread(img_path)
            if image is None: continue
            
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Eksekusi Augmentasi
            augmented = transform(image=image)['image']
            
            # Simpan hasil
            augmented_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
            new_name = f"aug_{generated_count}.jpg"
            cv2.imwrite(os.path.join(folder_to_augment, new_name), augmented_bgr)
            
            generated_count += 1
            pbar.update(1)
    pbar.close()

print(f"✅ Sekarang total gambar MalNutrisi: {len(os.listdir(folder_to_augment))}")

A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.


🔥 Menambah 526 gambar augmentasi untuk MalNutrisi...


100%|██████████| 526/526 [00:00<00:00, 1052.89it/s]


✅ Sekarang total gambar MalNutrisi: 550


In [9]:
import os

# Cek folder train saja untuk verifikasi akhir
train_path = os.path.join(DATASET_PATH, "train")
print("=== FINAL DATASET REPORT (TRAIN) ===")

total_data = 0
for label in os.listdir(train_path):
    label_dir = os.path.join(train_path, label)
    if os.path.isdir(label_dir):
        count = len(os.listdir(label_dir))
        print(f"✅ {label.ljust(15)}: {count} gambar")
        total_data += count

print("-" * 30)
print(f"TOTAL DATA TRAINING: {total_data} gambar")

=== FINAL DATASET REPORT (TRAIN) ===
✅ Deteksi-Stunting: 0 gambar
✅ Healthy        : 614 gambar
✅ MalNutrisi     : 550 gambar
✅ Stunting       : 531 gambar
------------------------------
TOTAL DATA TRAINING: 1695 gambar
